In [38]:
from moviepy import VideoClip
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import math

WIDTH = 1920
HEIGHT = 1080

DURATION = 60
FPS = 30

BG_COLOR = (30, 29, 28)
GRID_COLOR = (200, 195, 190)

HEX_SIZE = 120
LINE_WIDTH = 2

SHOW_NUMBERS = True

COLOR_A = (60, 75, 55)
COLOR_B = (95, 85, 65)

B_CELLS = {
    1, 2, 25, 49,
    *range(71, 78),
    13, 26, 14, 27, 37, 38, 39,
    50, 51, 52, 61, 62, 63, 64
}

ICON_CELLS = {3, 15, 28, 40, 53, 65, 78}

# ✔ bars только в клетках с движением
BAR_CELLS = {15, 53}

# ✔ параметры анимации
ICON_COUNT = 8

# ✔ параметры времени
t = 2  # игровых часов за секунду анимации
ct = 44  # время боя, игровых часов
ct2 = ct / t  # время боя, секунд анимации

# ✔ время отступления
rt = 26  # время отступления пехоты, игровых часов
rt2 = rt / t  # время отступления, секунд анимации

reactiontime = 24  # время реакции до начала отступления из котла
reactiontime2 = 24 / t  # время реакции до начала отступления из котла

# танки 
tankvelocity = 1/16
tankvelocity2 = tankvelocity * t
battlespeedreductionfactor = 0.35

# ✔ танки параметры движения
TANK_DISTANCE = 3  # 3 клетки путь

def tank_v1():
    return tankvelocity2 * battlespeedreductionfactor

def tank_v2():
    return tankvelocity2

def tank_D1():
    return tank_v1() * ct2

def tank_X():
    return min(TANK_DISTANCE, tank_D1())

def tank_total_time():
    D1 = tank_D1()
    D2 = max(0.0, TANK_DISTANCE - D1)
    return ct2 + (D2 / tank_v2() if tank_v2() > 0 else 0)

BAR_GREEN = (76, 175, 80)

BAR_WIDTH = 14
BAR_HEIGHT = 60
BAR_GAP = 6

ICON_OFFSET_X = -10
BAR_OFFSET_X = 30

ICON_PATH = "Inftransparent.png"
ICON = Image.open(ICON_PATH).convert("RGBA")

# ✔ танк
TANK_PATH = "Tank.png"
TANK_ICON = Image.open(TANK_PATH).convert("RGBA")


def hexagon_points(cx, cy, size):
    return [
        (
            cx + size * math.cos(math.radians(60 * i - 30)),
            cy + size * math.sin(math.radians(60 * i - 30))
        )
        for i in range(6)
    ]


def get_center(points):
    x = sum(p[0] for p in points) / len(points)
    y = sum(p[1] for p in points) / len(points)
    return x, y


def icon_positions(cx, cy, size):
    w = size * 1.2
    h = size * 1.2

    if ICON_COUNT == 12:
        cols = 4
        rows = 3
    else:
        cols = 4
        rows = 2

    positions = []

    for r in range(rows):
        for c in range(cols):
            x = cx - w / 2 + (c + 0.5) * (w / cols)
            y = cy - h / 2 + (r + 0.5) * (h / rows)
            positions.append((x, y))

    return positions[:ICON_COUNT]


# расположение танков
def tank_positions(cx, cy, size):
    w = size * 1.2
    h = size * 1.2
    y = cy
    offset_x = w * 0.3

    left = (cx - offset_x, y)
    right = (cx + offset_x, y)

    return [left, right]


def draw_bar(draw, x, y, width, height, fill_ratio, fill_color):
    draw.rectangle(
        [x, y, x + width, y + height],
        outline=GRID_COLOR,
        width=1
    )

    fill_h = int(height * fill_ratio)

    draw.rectangle(
        [
            x,
            y + (height - fill_h),
            x + width,
            y + height
        ],
        fill=fill_color
    )


def lerp(a, b, t):
    return a + (b - a) * t


# ✔ события движения
step_time = ct2 / ICON_COUNT

MOVES = []

for i in range(ICON_COUNT):

    MOVES.append({
        "icon": i,
        "from_cell": 15,
        "to_cell": 4,
        "to_index": i,
        "t_start": (i + 1) * step_time,
        "t_end": (i + 1) * step_time + rt2,
        "duration": rt2
    })

for i in range(ICON_COUNT):

    MOVES.append({
        "icon": i,
        "from_cell": 53,
        "to_cell": 54,
        "to_index": i,
        "t_start": (i + 1) * step_time,
        "t_end": (i + 1) * step_time + rt2,
        "duration": rt2
    })

for i in range(ICON_COUNT):

    MOVES.append({
        "icon": i,
        "from_cell": 28,
        "to_cell": 30,
        "to_index": i,
        "t_start": reactiontime2,
        "t_end": reactiontime2 + 2 * rt2,
        "duration": 2 * rt2
    })

for i in range(ICON_COUNT):

    MOVES.append({
        "icon": i,
        "from_cell": 40,
        "to_cell": 17,
        "to_index": i,
        "t_start": reactiontime2,
        "t_end": reactiontime2 + 2 * rt2,
        "duration": 2 * rt2
    })


# ✔ танки (старт → конец)
TANK_MOVES = {
    14: 17,
    27: 17,
    52: 30,
    64: 30
}


def tank_progress(t_sec):

    v1 = tank_v1()
    v2 = tank_v2()

    D1 = v1 * ct2

    if t_sec <= ct2:
        return (v1 * t_sec) / TANK_DISTANCE if TANK_DISTANCE > 0 else 0

    D2 = max(0.0, TANK_DISTANCE - D1)
    t2 = D2 / v2 if v2 > 0 else 0

    if t_sec >= ct2 + t2:
        return 1.0

    return (D1 + v2 * (t_sec - ct2)) / TANK_DISTANCE


def make_frame(t_sec):

    img = Image.new("RGB", (WIDTH, HEIGHT), BG_COLOR)
    draw = ImageDraw.Draw(img)

    hex_width = math.sqrt(3) * HEX_SIZE
    x_step = hex_width
    y_step = 1.5 * HEX_SIZE

    cols = int(WIDTH / x_step) + 3
    rows = int(HEIGHT / y_step) + 3

    icon_size = int(HEX_SIZE * 0.35)
    icon_resized = ICON.resize((icon_size, icon_size))

    tank_size = icon_size * 2
    tank_resized = TANK_ICON.resize((tank_size, tank_size))

    cell_centers = {}
    cell_id = 1

    for row in range(rows):
        for col in range(cols):

            x = col * x_step
            y = row * y_step

            if row % 2 == 1:
                x += x_step / 2

            points = hexagon_points(x, y, HEX_SIZE)
            cx, cy = get_center(points)

            cell_centers[cell_id] = (cx, cy)

            draw.polygon(
                points,
                fill=COLOR_B if cell_id in B_CELLS else COLOR_A,
                outline=GRID_COLOR,
                width=LINE_WIDTH
            )

            if cell_id in BAR_CELLS:

                bar_x = cx + HEX_SIZE * 0.35 + BAR_OFFSET_X
                bar_y = cy - BAR_HEIGHT / 2

                green_fill = max(0.0, 1.0 - (t_sec / ct2))

                draw_bar(draw, bar_x, bar_y,
                         BAR_WIDTH, BAR_HEIGHT,
                         green_fill, BAR_GREEN)

            cell_id += 1

    moving_from_cells = {m["from_cell"] for m in MOVES}

    for cid, (cx, cy) in cell_centers.items():

        if cid not in ICON_CELLS:
            continue

        positions = icon_positions(cx, cy, HEX_SIZE)

        if cid in moving_from_cells:
            for local_index in range(ICON_COUNT):

                in_motion = False
                for m in MOVES:
                    if m["from_cell"] == cid and m["to_index"] == local_index:
                        if t_sec < m["t_start"]:
                            px, py = positions[local_index]
                            img.paste(icon_resized,
                                      (int(px - icon_size / 2 + ICON_OFFSET_X),
                                       int(py - icon_size / 2)),
                                      icon_resized)
                        in_motion = True
                        break

                if not in_motion:
                    px, py = positions[local_index]
                    img.paste(icon_resized,
                              (int(px - icon_size / 2 + ICON_OFFSET_X),
                               int(py - icon_size / 2)),
                              icon_resized)

        else:
            for px, py in positions:
                img.paste(icon_resized,
                          (int(px - icon_size / 2 + ICON_OFFSET_X),
                           int(py - icon_size / 2)),
                          icon_resized)

    for m in MOVES:

        if not (m["t_start"] <= t_sec <= m["t_end"]):
            continue

        progress = (t_sec - m["t_start"]) / m["duration"]

        from_positions = icon_positions(*cell_centers[m["from_cell"]], HEX_SIZE)
        to_positions = icon_positions(*cell_centers[m["to_cell"]], HEX_SIZE)

        sx, sy = from_positions[m["to_index"]]
        tx, ty = to_positions[m["to_index"]]

        x = lerp(sx, tx, progress)
        y = lerp(sy, ty, progress)

        img.paste(icon_resized,
                  (int(x - icon_size / 2 + ICON_OFFSET_X),
                   int(y - icon_size / 2)),
                  icon_resized)

    for m in MOVES:

        if t_sec > m["t_end"]:

            cx, cy = cell_centers[m["to_cell"]]

            px, py = icon_positions(cx, cy, HEX_SIZE)[m["to_index"]]

            img.paste(icon_resized,
                      (int(px - icon_size / 2 + ICON_OFFSET_X),
                       int(py - icon_size / 2)),
                      icon_resized)

    for from_cell, to_cell in TANK_MOVES.items():

        from_cx, from_cy = cell_centers[from_cell]
        to_cx, to_cy = cell_centers[to_cell]

        from_pos = tank_positions(from_cx, from_cy, HEX_SIZE)
        to_pos = tank_positions(to_cx, to_cy, HEX_SIZE)

        progress = tank_progress(t_sec)

        for i in range(2):

            sx, sy = from_pos[i]
            tx, ty = to_pos[i]

            x = lerp(sx, tx, progress)
            y = lerp(sy, ty, progress)

            img.paste(
                tank_resized,
                (int(x - tank_size / 2),
                 int(y - tank_size / 2)),
                tank_resized
            )

    cell_id = 1
    font = ImageFont.load_default()

    for row in range(rows):
        for col in range(cols):

            x = col * x_step
            y = row * y_step

            if row % 2 == 1:
                x += x_step / 2

            points = hexagon_points(x, y, HEX_SIZE)
            cx, cy = get_center(points)

            text = str(cell_id)

            bbox = draw.textbbox((0, 0), text, font=font)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]

            draw.text(
                (cx - tw / 2, cy - th / 2),
                text,
                fill=(235, 232, 228),
                font=font
            )

            cell_id += 1

    return np.array(img)


clip = VideoClip(make_frame, duration=DURATION)

clip.write_videofile(
    "hex_static.mp4",
    fps=FPS,
    codec="libx264",
    bitrate="8M"
)

frame_index:   5%|▍         | 84/1800 [50:49<02:40, 10.68it/s, now=None]

MoviePy - Building video hex_static.mp4.
MoviePy - Writing video hex_static.mp4



frame_index:   5%|▍         | 84/1800 [54:34<02:40, 10.68it/s, now=None]

MoviePy - Done !
MoviePy - video ready hex_static.mp4
